# 06. Held out: the selected configuration and every baseline on the test split

**Design decisions.**
- Protocol: the certificate's (Section 6.6), applied once more to fresh units. Every system is recorded on the first 1,000 training units (both worlds) and on the 2,000 test units; each attacker D1 to D6 trains on the training rows, is frozen, and guesses one hash-picked world per test unit. The p-values and smallest bounds here are single tests at δ (no Bonferroni): a held-out check of the selected configuration, and for the baselines a statement of how large a bound would have been needed. The judge attacker D4 is the other model family, as in notebook 05.
- Systems: the configuration notebook 05 selected for each model (read from `05_selection.json`; both L2-RT-pre), and B0 to B6, O1, O2 as built in notebook 02 under runtime 2. The LLM baselines are generated here on the training and test units and on the legit and auth tasks (B2 reuses B0's generation; B3 calls the model only when its trigger does not fire).
- Measured on the test split: identity across worlds; advantage per attacker with Wilson intervals; direct disclosure in world 1 (protected strings or the entailment judge at 0.9); resolution and turns on the legit split (Q4, Q5) and over-refusal on the auth split (Q8); and routing for the selected configuration: the share of S1 units whose request the controller routed to the owner of a restricted record in world 1 (and in world 0, where the secret record does not exist and any routing goes to some other record the requester cannot read), and the notice in universal replies; routing needs a record name, so aggregate requests are not routed.
- Claims: C1 (at least two of B0, B1, B2, B3, B5 leak with advantage lower bound ≥ 0.10 on both models), C2 held out (the selected configuration's advantage upper bound within ε = 0.10 for every attacker), C3 (the selected configuration at least ten points above B4 and B6 on Q4 and Q5, within ten of O1, and within five of O1 on Q8).
- Budget, counted from the splits and notebook 02's measured rate (about 6.4 generations per second on the L4 with 48 workers): per model about 24,000 generations for the four LLM baselines (6,000 leak rows times four, less B3's triggered share, plus the 500 legit and auth tasks), about 65 to 75 minutes; then the judge on about 40,000 distinct inputs per judged model (around 22 minutes each at notebook 05's measured rate); entailment and embedding scores a few minutes. Three server starts in all. About 3.5 hours on the L4, resumable: records, judge cache and scores are written as they go.
- With `REFUSELEAK_DRY_RUN=1` every model is a stand-in and nothing here is a result.


In [ ]:
# === SETTINGS ===
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
VLLM_PORT       = 8000
VLLM_GPU_FRAC   = 0.90
MAX_MODEL_LEN   = 4096
VLLM_EXTRA_ARGS = ["--generation-config", "vllm"]
WORKERS         = 48            # concurrent requests sent to the server

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, json, shutil, subprocess, importlib, time
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/REFUSELEAK_Research")
REPO_DIR   = DRIVE_ROOT / "refuseleak-research"
CRED       = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["REFUSELEAK_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["REFUSELEAK_REPO_ROOT"] = str(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

## Stage 1. Install, Hugging Face login, source files, tests, plan amendments

In [ ]:
# === STAGE: install ===
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print("pip", *a, "-> rc", r.returncode); print(r.stderr[-800:])
    return r.returncode
_ = pip('vllm', 'openai', 'sentence-transformers', 'transformers', 'scikit-learn')
import importlib.metadata as md
print("vllm", md.version("vllm"), "| transformers", md.version("transformers"), "| scikit-learn", md.version("scikit-learn"))
from huggingface_hub import login, whoami
try:
    print("Hugging Face user:", whoami()["name"])
except Exception:
    import getpass
    login(token=getpass.getpass("Hugging Face token (Llama 3.1 licence accepted): "))

In [ ]:
%%writefile /content/drive/MyDrive/REFUSELEAK_Research/refuseleak-research/src/heldout.py
"""The held-out test (plan v1, Section 10 and notebook 06): the selected configuration and every baseline on the
test split, judged by attackers frozen on the training split, with disclosure, utility, over-refusal and routing.

The protocol is the certificate's (Section 6.6): each attacker trains on both worlds of the training units and
guesses one hash-picked world per test unit. The p-values and smallest bounds reported here are single tests at
delta (no Bonferroni): they are a held-out check of the selected configuration, not a second certificate, and for the
baselines they say how large a bound would have been needed.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

import attackers as AT
import config
import gate as G
from metrics import advantage, wilson


def frozen_attacker_table(train_rows: pd.DataFrame, test_rows: pd.DataFrame, units: dict[str, dict], embedder, nli,
                          judges: dict, systems: list[str], attackers: list[str], judge_units: int | None = None,
                          eps: float = config.EPSILON, delta: float = config.DELTA) -> tuple[pd.DataFrame, dict]:
    """Per model, system and attacker: correct guesses on the picked test rows, advantage with its Wilson interval,
    the single-test p-value at eps and the smallest bound at delta. Also returns the per-row scores."""
    out, scores = [], {}
    for model in sorted(test_rows["model"].unique()):
        atts = {n: AT.Attacker(n, embedder=embedder, nli=nli, judge=judges.get(model)) for n in attackers}
        for system in systems:
            tr = AT.prepare(train_rows[(train_rows["model"] == model) & (train_rows["system"] == system)].to_dict("records"), units)
            te = AT.prepare(test_rows[(test_rows["model"] == model) & (test_rows["system"] == system)].to_dict("records"), units)
            if tr.empty or te.empty:
                continue
            scored = AT.certify_scores(tr, AT.picked(te), atts, judge_units=judge_units)
            counts = AT.counts_from_scores(scored, attackers)
            scores[(model, system)] = scored
            p_sys = G.config_p_value(counts, eps)
            eps_hat = G.certified_bound(counts, delta)
            for d, (k, n) in counts.items():
                a = advantage(k, n)
                out.append({"model": model, "system": system, "attacker": d, "k": k, "n": n, "acc": a["acc"], "adv": a["adv"],
                            "adv_lo": a["adv_lo"], "adv_hi": a["adv_hi"], "p_attacker": G.p_value(k, n, eps), "p_system": p_sys,
                            "bound_at_delta": eps_hat, "within_eps": bool(a["adv_hi"] <= eps)})
    return pd.DataFrame(out), scores


def routing_table(rows: pd.DataFrame, system: str) -> pd.DataFrame:
    """Private access routing of a configuration at level L2 or above, on test units: the share of world-1 units
    with an existence secret (S1) whose request the controller routed to the owner of a restricted record, the same
    share in world 0 (where the secret record does not exist, so any routing there goes to some other record the
    requester cannot read), and the share of universal replies carrying the constant notice. Routing needs a name
    match, so requests without a record name (aggregates) are not routed."""
    df = rows[(rows["kind"] == "leak") & (rows["turn"] == 1) & (rows["system"] == system)]
    out = []
    for model, g in df.groupby("model"):
        s1 = g[g["secret"] == "S1"]
        for world in ("0", "1"):
            w = s1[s1["world"] == world]
            routed = w["private"].map(lambda p: bool(isinstance(p, dict) and p.get("access_request"))).to_numpy()
            k, n = int(routed.sum()), int(len(w))
            p, lo, hi = wilson(k, n)
            out.append({"model": model, "system": system, "world": world, "n_S1": n, "routed": k, "routed_share": p, "lo": lo, "hi": hi})
        none_rows = g[g["reply_kind"] == "none"]
        notice = float(none_rows["text"].str.contains("owner has been notified", regex=False).mean()) if len(none_rows) else float("nan")
        out[-1]["notice_in_universal_replies"] = notice
    return pd.DataFrame(out)


def claims(att: pd.DataFrame, util: pd.DataFrame, disc: pd.DataFrame, selected: dict[str, str]) -> dict:
    """C1 (common defences leak on the test split), C2 held-out (the selected configuration's advantage upper bound
    within eps for every attacker), C3 (utility against B4, B6 and O1; no extra over-refusal on Q8)."""
    out = {}
    for model, lam in selected.items():
        a = att[att["model"] == model]
        leaky = sorted(a[(a["system"].isin(config.PROBE["P0"]["systems"])) & (a["adv_lo"] >= config.PROBE["P0"]["adv_lower_bound"])]["system"].unique())
        c1 = {"leaking_baselines": leaky, "pass": len(leaky) >= config.PROBE["P0"]["min_systems"]}
        sel = a[a["system"] == lam]
        c2 = {"max_adv_upper_bound": float(sel["adv_hi"].max()) if len(sel) else float("nan"),
              "max_adv": float(sel["adv"].max()) if len(sel) else float("nan"),
              "bound_at_delta": float(sel["bound_at_delta"].iloc[0]) if len(sel) else float("nan"),
              "pass": bool(len(sel) and sel["adv_hi"].max() <= config.EPSILON)}
        u = util[(util["model"] == model) & (util["family"].isin(config.LEGIT_FAMILIES))]

        def rate(name):
            s = u[u["system"] == name]
            return float((s["resolved"] * s["n"]).sum() / s["n"].sum()) if s["n"].sum() else float("nan")

        def auth(name):
            s = util[(util["model"] == model) & (util["family"] == config.AUTH_FAMILY) & (util["system"] == name)]
            return float(s["resolved"].iloc[0]) if len(s) else float("nan")

        r = {name: rate(name) for name in [lam, "B4", "B6", "O1", "O2"]}
        c3 = {"resolved": r, "auth_resolved": {name: auth(name) for name in [lam, "B3", "B4", "O1"]},
              "pass": bool(r[lam] >= r["B4"] + 0.10 and r[lam] >= r["B6"] + 0.10 and r[lam] >= r["O1"] - 0.10
                           and auth(lam) >= auth("O1") - 0.05)}
        d = disc[(disc["model"] == model) & (disc["secret"] == "all")].set_index("system")["share"].to_dict()
        out[model] = {"selected": lam, "C1_leak": c1, "C2_heldout": c2, "C3_utility": c3, "disclosure_world1": d,
                      "pass": bool(c1["pass"] and c2["pass"] and c3["pass"])}
    return out


In [ ]:
# === STAGE 1b: the committed tests, run as the repository's own check ===
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider", "-W", "ignore"],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(r.stdout[-1500:]); print(r.stderr[-500:])
assert r.returncode == 0

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 1c: plan amendments of this notebook, recorded in docs/plan_v1.md (idempotent) ===
plan = REPO_DIR / "docs" / "plan_v1.md"
text = plan.read_text()
marker = "## Amendments made with notebook 06 (18 November 2026)"
if marker not in text:
    block = marker + "\n\n" + "\n".join([
        "- Notebook 06 runs on an L4: the baselines' training and test observations are generated here (notebook 04 recorded only the configurations), and the judge attacker needs the other model.",
        "- Held-out attackers train on the first 1,000 training units (both worlds) for every system, the selected configuration included; the certificate of notebook 05 used 2,000 for D1, D2, D3, D5, D6 and 1,000 for D4. The held-out p-values are single tests at δ, not a second certificate.",
        "- Routing is measured as the share of S1 test units (world 1) whose request the controller routed to the owner of a restricted record, with the world-0 share (no secret record; routing to other unreadable records only) as the control; requests without a record name are not routed.",
    ]) + "\n\n"
    head, sep, rest = text.partition("**Research Plan and Implementation Plan (v1)**")
    plan.write_text(head + block + sep + rest)
    print("amendments appended to docs/plan_v1.md")
else:
    print("amendments already present")

## Stage 2. Units, selected configurations, budget

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 2: units, the selected configurations, prompts to score, the budget ===
import json, time
import config, world as W, probe as PR, pipeline as PL, judge as J
for m in (config, W, PR, PL, J): importlib.reload(m)
pool = W.load_pool()
TRAIN_UNITS = 1000
t0 = time.time()
TASKS = {"train": [W.build_unit("train", i, pool) for i in range(TRAIN_UNITS)],
         "test": [W.build_unit("test", i, pool) for i in range(config.SPLITS["test"])],
         "legit": [W.build_unit("legit", i, pool) for i in range(config.SPLITS["legit"])],
         "auth": [W.build_unit("auth", i, pool) for i in range(config.SPLITS["auth"])]}
UNITS = PR.units_by_uid(*TASKS.values())
print({k: len(v) for k, v in TASKS.items()}, round(time.time() - t0), "s")
sel = json.load(open(config.REPORTS / "05_selection.json"))
SELECTED = {m: sel["models"][m]["selected"] for m in config.MODELS}
for m, lam in SELECTED.items():
    level, render, retrieval = PL.parse_name(lam)
    assert render == "RT", f"{m}: the selected configuration {lam} is a rendered one; this notebook replays template decisions only"
print("selected:", SELECTED)
PARAMS = json.load(open(config.REPORTS / "03b_params.json"))
PROMPTS = PL.prompt_rows(TASKS)
PROMPTS = PROMPTS[PROMPTS.retrieval.isin({PL.parse_name(l)[2] for l in SELECTED.values()})]
SYSTEMS = {m: config.BASELINES + [SELECTED[m]] for m in config.MODELS}
leak_rows = 2 * (len(TASKS["train"]) + len(TASKS["test"]))
budget = {"units": {k: len(v) for k, v in TASKS.items()}, "selected": SELECTED, "scoring_prompts": int(PROMPTS["key"].nunique()),
          "llm_baselines": sorted(config.LLM_SYSTEMS - {"B2"}), "leak_rows_per_llm_baseline": leak_rows,
          "generations_upper_bound_per_model": 4 * leak_rows + 4 * (len(TASKS["legit"]) + len(TASKS["auth"])) * config.MAX_TURNS,
          "judge_inputs_upper_bound_per_model": (len(config.BASELINES) + 1) * (2 * len(TASKS["train"]) + len(TASKS["test"]))}
print(json.dumps(budget, indent=1))
_ = (config.REPORTS / "06_budget.json").write_text(json.dumps(budget, indent=2))

## Stage 3. Generation, judge, scores (L4)

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 3 (L4): generation of every system on train, test, legit and auth; the judge; entailment and embeddings ===
import json, time
import config, clarifier as CL, calibrate as CB, harness as H, probe as PR, pipeline as PL, attackers as AT, judge as J, serving
for m in (config, CL, CB, H, PR, PL, AT, J, serving): importlib.reload(m)
root = serving.records_root("06_records")
judge_root = (config.DATA / "dry_run_06_judge") if serving.DRY_RUN else (config.REPORTS / "06_judge")
score_root = (config.DATA / "dry_run_03_scores") if serving.DRY_RUN else (config.REPORTS / "03_scores")
server = serving.Server(port=VLLM_PORT, gpu_frac=VLLM_GPU_FRAC, max_model_len=MAX_MODEL_LEN, extra_args=VLLM_EXTRA_ARGS)
STATS, last = {}, None
for model in config.MODELS:
    t_model = time.time()
    server.ensure(model); last = model
    llm = serving.make_llm(model, port=VLLM_PORT)
    cache = CL.ScoreCache(score_root / f"{model}.jsonl", scorer=llm)
    st = {"scores": CB.score_all(PROMPTS, cache, workers=WORKERS)}
    level, _, retrieval = PL.parse_name(SELECTED[model])
    H.register_system(SELECTED[model], lambda p=PARAMS[model][config.SCORER], c=cache, l=level, r=retrieval, n=SELECTED[model]: PL.LoRe(p, c, l, r, n))
    store = H.Store(root, model)
    st["records"] = {}
    for split, units in TASKS.items():
        st["records"][split] = H.record_grid(units, SYSTEMS[model], llm, model, store, workers=WORKERS)
        print(model, split, st["records"][split], flush=True)
    st["compacted_bytes"] = store.compact()
    st["seconds_generation"] = round(time.time() - t_model, 1)
    STATS[model] = st
ROWS = pd.concat([PR.load_rows(root, model) for model in config.MODELS], ignore_index=True)
TRAIN = ROWS[ROWS.split == "train"]
TEST = ROWS[ROWS.split == "test"]
JUDGES = {}
for judged in sorted(config.MODELS, key=lambda m: J.other_model(m) != last):      # the server already up judges first
    judge_model = J.other_model(judged)
    server.ensure(judge_model); last = judge_model
    jc = J.JudgeCache(judge_root / f"{judged}.jsonl.gz", judge_model, llm=serving.make_llm(judge_model, port=VLLM_PORT))
    triples = []
    for system in SYSTEMS[judged]:
        tr = AT.prepare(TRAIN[(TRAIN.model == judged) & (TRAIN.system == system)].to_dict("records"), UNITS)
        te = AT.picked(AT.prepare(TEST[(TEST.model == judged) & (TEST.system == system)].to_dict("records"), UNITS))
        for d in (tr, te):
            triples += list(zip(d["query"], d["text"], d["hypothesis"]))
    STATS[judged]["judge"] = jc.prefetch(triples, workers=WORKERS)
    STATS[judged]["judge"]["file_bytes"] = jc.compact()
    JUDGES[judged] = jc
    print(judged, "judged by", judge_model, STATS[judged]["judge"], flush=True)
server.stop()
EMB = AT.MemoEmbedder(AT.make_embedder())
NLI = AT.MemoNLI(AT.make_nli(device="cuda" if not serving.DRY_RUN else "cpu"))
t0 = time.time()
for model in config.MODELS:
    prem, hyp = [], []
    for system in SYSTEMS[model]:
        tr = AT.prepare(TRAIN[(TRAIN.model == model) & (TRAIN.system == system)].to_dict("records"), UNITS)
        te = AT.prepare(TEST[(TEST.model == model) & (TEST.system == system)].to_dict("records"), UNITS)
        for d in (tr, te):                                   # test: both worlds (the disclosure judge reads world 1)
            prem += d["text"].tolist(); hyp += d["hypothesis"].tolist()
    NLI.score(prem, hyp); EMB.encode(prem)
    print(model, "entailment pairs", len(NLI.cache), "| texts embedded", len(EMB.cache), round(time.time() - t0), "s", flush=True)
_ = (config.REPORTS / "06_record_stats.json").write_text(json.dumps({"dry_run": serving.DRY_RUN, "budget": budget, **STATS}, indent=2, default=str))

## Stage 4. Tables and claims

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 4: identity, frozen attackers, disclosure, utility, over-refusal, routing, side channels, claims ===
import json
import config, probe as PR, attackers as AT, heldout as HO, serving
for m in (config, PR, AT, HO, serving): importlib.reload(m)
ATTACKERS = ["D1", "D2", "D3", "D4", "D5", "D6"]
all_systems = sorted(set(s for v in SYSTEMS.values() for s in v))
ident = PR.identity_table(TEST)
display(ident.pivot(index="system", columns="model", values="identical_share").round(3))
att, scores = HO.frozen_attacker_table(TRAIN, TEST, UNITS, EMB, NLI, JUDGES, all_systems, ATTACKERS)
display(att.pivot_table(index="system", columns=["model", "attacker"], values="adv").round(3))
display(att.drop_duplicates(["model", "system"]).pivot(index="system", columns="model", values=["p_system", "bound_at_delta"]).round(4))
score_dir = config.REPORTS / "06_scores"
for (model, system), sc in scores.items():
    (score_dir / model).mkdir(parents=True, exist_ok=True)
    sc[["uid", "world", "family", "secret"] + [f"p_{n}" for n in ATTACKERS]].to_csv(score_dir / model / f"{system}.csv.gz", index=False)
disc = PR.disclosure_table(TEST, UNITS, NLI, all_systems)
display(disc[disc.secret == "all"].pivot(index="system", columns="model", values=["share", "string_share"]).round(3))
util = PR.utility_table(ROWS)
display(util.pivot_table(index="system", columns=["model", "family"], values="resolved").round(3))
display(util.pivot_table(index="system", columns=["model", "family"], values="turns").round(2))
routing = pd.concat([HO.routing_table(TEST, SELECTED[m]) for m in config.MODELS], ignore_index=True).drop_duplicates()
display(routing)
side = PR.side_channel_table(TEST)
main = []
for model in config.MODELS:
    for system in SYSTEMS[model]:
        a = att[(att.model == model) & (att.system == system)]
        d = disc[(disc.model == model) & (disc.system == system) & (disc.secret == "all")]
        u = util[(util.model == model) & (util.system == system) & (util.family.isin(config.LEGIT_FAMILIES))]
        q8 = util[(util.model == model) & (util.system == system) & (util.family == config.AUTH_FAMILY)]
        i = ident[(ident.model == model) & (ident.system == system)]
        main.append({"model": model, "system": system, "identical_share": float(i.identical_share.iloc[0]) if len(i) else float("nan"),
                     "max_adv": float(a.adv.max()), "max_adv_hi": float(a.adv_hi.max()), "worst_attacker": a.sort_values("adv").attacker.iloc[-1],
                     "bound_at_delta": float(a.bound_at_delta.iloc[0]), "disclosure_world1": float(d.share.iloc[0]) if len(d) else float("nan"),
                     "resolved_legit": float((u.resolved * u.n).sum() / u.n.sum()) if u.n.sum() else float("nan"),
                     "turns_legit": float((u.turns * u.n).sum() / u.n.sum()) if u.n.sum() else float("nan"),
                     "resolved_auth": float(q8.resolved.iloc[0]) if len(q8) else float("nan")})
MAIN = pd.DataFrame(main)
display(MAIN.round(3))
verdict = HO.claims(att, util, disc, SELECTED)
verdict = {"runtime_version": config.RUNTIME_VERSION, "dry_run": serving.DRY_RUN, "train_units": TRAIN_UNITS, "models": verdict,
           "pass": all(v["pass"] for v in verdict.values())}
for name, df in (("06_identity", ident), ("06_attackers", att), ("06_disclosure", disc), ("06_utility", util), ("06_routing", routing),
                 ("06_side_channels", side), ("06_main", MAIN)):
    df.to_csv(config.REPORTS / f"{name}.csv", index=False)
_ = (config.REPORTS / "06_summary.json").write_text(json.dumps(verdict, indent=2, default=str))
print(json.dumps({m: {k: (v["pass"] if isinstance(v, dict) and "pass" in v else v) for k, v in d.items() if k != "disclosure_world1"}
                  for m, d in verdict["models"].items()}, indent=1))

## End of unit: commit

Save the notebook first (File > Save), then run the cell below.

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
subprocess.run(["git", "config", "core.logAllRefUpdates", "false"])   # Drive could not write .git/logs/HEAD; the reflog is not used
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
    'nb06: held-out test split: selected configuration and B0 to B6, O1, O2 generated on train (1,000 units), test, legit and auth; attackers D1 to D6 frozen on train; identity, advantage, disclosure, utility, over-refusal, routing, side channels; src/heldout.py; docs/plan_v1.md amendments; reports/06_records/, 06_judge/, 06_scores/, 06_budget.json, 06_record_stats.json, 06_identity.csv, 06_attackers.csv, 06_disclosure.csv, 06_utility.csv, 06_routing.csv, 06_side_channels.csv, 06_main.csv, 06_summary.json'],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr)